# Neutron Capture Probability Calculator

This notebook calculates the **probability of neutron capture** as a function of neutron energy for different materials, taking into account material properties and geometry.

## Features

⚛️ **Comprehensive Nuclear Database**: Includes thermal neutron cross-sections, resonance parameters, and fast neutron data for key isotopes

🎛️ **Configurable Parameters**: Easy selection of materials, density, thickness, and temperature

📊 **Multiple Plot Types**: Cross-sections, capture probabilities, thickness dependence, temperature effects

🌡️ **Temperature Effects**: Accurate 1/v law implementation with thermal motion corrections

🎯 **Interactive Analysis**: Functions for specific calculations and material comparisons

## Key Physics

The neutron capture probability is governed by:

$$P = 1 - e^{-\Sigma t}$$

Where:
- $\Sigma = n \sigma(E)$ = macroscopic cross-section (cm⁻¹)
- $n = \frac{\rho N_A \text{abundance}}{A}$ = number density (atoms/cm³)
- $\sigma(E)$ = energy-dependent microscopic cross-section (barns)
- $t$ = material thickness (cm)

### Cross-Section Models:
- **Thermal Region** (E < 0.5 eV): $\sigma(E) = \sigma_{th} \sqrt{\frac{E_{th}}{E}}$ (1/v law)
- **Resonance Region** (0.5 eV - 10 keV): Breit-Wigner resonance peaks
- **Fast Region** (E > 10 keV): Slowly varying cross-sections

---

**📝 Instructions**: Modify the configuration parameters, then run all cells to generate comprehensive neutron capture analysis!

In [1]:
"""
Neutron Capture Probability Calculator
=====================================

This notebook provides an interactive interface for neutron capture analysis
using the particle detectors physics library.
"""

import numpy as np
import matplotlib.pyplot as plt

# Import our physics library
from lib import (
    nuclear_db,
    calculate_capture_probability,
    calculate_transmission, 
    calculate_specific_neutron_case,
    compare_materials_at_energy
)

print("✓ Neutron Capture Calculator loaded successfully!")
print(f"  - Nuclear database: {len(nuclear_db.nuclear_data)} isotopes available")
print("  - Ready for interactive neutron capture analysis")

ModuleNotFoundError: No module named 'scipy'

In [ ]:
# ===================================================================
# CONFIGURATION SECTION - MODIFY THESE PARAMETERS
# ===================================================================

# Select materials/isotopes to analyze
SELECTED_MATERIALS = [
    'B-10',      # Boron-10 (high thermal capture cross-section)
    'Cd-113',    # Cadmium-113 (strong neutron absorber)  
    'Gd-155',    # Gadolinium-155 (excellent thermal absorber)
    'Gd-157',    # Gadolinium-157 (highest known thermal cross-section)
    'Li-6',      # Lithium-6 (neutron converter)
    'He-3',      # Helium-3 (neutron detector gas)
    'U-235',     # Uranium-235 (fissile material)
    'H-1',       # Hydrogen (moderator)
]

# Material properties to vary
MATERIAL_DENSITIES = {
    'B-10': 2.34,       # g/cm³ (boron carbide)
    'Cd-113': 8.65,     # g/cm³ (metallic cadmium)
    'Gd-155': 7.90,     # g/cm³ (metallic gadolinium)  
    'Gd-157': 7.90,     # g/cm³ (metallic gadolinium)
    'Li-6': 0.534,      # g/cm³ (metallic lithium)
    'He-3': 0.000178,   # g/cm³ (gas at STP)
    'U-235': 19.05,     # g/cm³ (metallic uranium)
    'H-1': 1.0,         # g/cm³ (water)
}

# Geometric parameters
THICKNESSES = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]  # cm
DEFAULT_THICKNESS = 1.0  # cm

# Energy range for neutrons
ENERGY_MIN = 1e-9    # eV (ultra-cold neutrons)
ENERGY_MAX = 1e7     # eV (fast neutrons)
NUM_POINTS = 1000

# Temperature for thermal neutron calculations
TEMPERATURE = 300    # Kelvin (room temperature)

# Reference energy for normalization (thermal energy at room temp)
THERMAL_ENERGY = 0.0253  # eV at 20°C

print(f"Selected materials: {SELECTED_MATERIALS}")  
print(f"Energy range: {ENERGY_MIN:.1e} - {ENERGY_MAX:.1e} eV")
print(f"Default thickness: {DEFAULT_THICKNESS} cm")
print(f"Temperature: {TEMPERATURE} K (thermal energy = {THERMAL_ENERGY} eV)")

Selected materials: ['B-10', 'Cd-113', 'Gd-155', 'Gd-157', 'Li-6', 'He-3', 'U-235', 'H-1']
Energy range: 1.0e-09 - 1.0e+07 eV
Default thickness: 1.0 cm
Temperature: 300 K (thermal energy = 0.0253 eV)


In [ ]:
# ===================================================================
# NUCLEAR DATABASE INFO
# ===================================================================

# Display available isotopes from the imported nuclear database
nuclear_db.list_available_isotopes()

In [ ]:
# ===================================================================
# CALCULATION FUNCTIONS (from lib.py)
# ===================================================================

# All heavy calculation functions are imported from lib.py:
# - calculate_capture_probability()
# - calculate_transmission()
# - calculate_specific_neutron_case() 
# - compare_materials_at_energy()

print("✓ Neutron capture calculation functions loaded from lib.py")

In [ ]:
# ===================================================================
# PLOTTING FUNCTIONS
# ===================================================================

def plot_cross_sections(materials_list, energies, temperature_k=TEMPERATURE):
    """Plot neutron capture cross-sections vs energy for different isotopes."""
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(materials_list)))
    
    for isotope, color in zip(materials_list, colors):
        try:
            cross_section = nuclear_db.calculate_capture_cross_section(
                isotope, energies, temperature_k)
            
            # Plot only positive cross-sections
            valid_mask = cross_section > 0
            if np.any(valid_mask):
                plt.loglog(energies[valid_mask], cross_section[valid_mask],
                          label=isotope, color=color, linewidth=2.5)
        except Exception as e:
            print(f"Warning: Could not plot {isotope}: {e}")
    
    plt.xlabel('Neutron Energy (eV)', fontsize=14)
    plt.ylabel('Capture Cross-Section (barns)', fontsize=14)
    plt.title('Neutron Capture Cross-Sections vs Energy', fontsize=16)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_capture_probabilities(materials_list, energies, thickness_cm=DEFAULT_THICKNESS,
                             temperature_k=TEMPERATURE):
    """Plot capture probabilities vs energy for different materials."""
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(materials_list)))
    
    for isotope, color in zip(materials_list, colors):
        try:
            density = MATERIAL_DENSITIES.get(isotope, 1.0)
            prob, xs, macro_xs = calculate_capture_probability(
                energies, isotope, nuclear_db, density, thickness_cm, temperature_k)
            
            # Plot all probabilities
            plt.loglog(energies, prob, label=f'{isotope} ({thickness_cm} cm)',
                      color=color, linewidth=2.5)
            
        except Exception as e:
            print(f"Warning: Could not plot {isotope}: {e}")
    
    plt.xlabel('Neutron Energy (eV)', fontsize=14)
    plt.ylabel('Capture Probability', fontsize=14)
    plt.title(f'Neutron Capture Probability vs Energy (thickness = {thickness_cm} cm)', fontsize=16)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.ylim(1e-6, 1)
    plt.tight_layout()
    plt.show()

def plot_thickness_dependence(isotope, energies_to_plot, thickness_range=None,
                            temperature_k=TEMPERATURE):
    """Plot capture probability vs thickness for different neutron energies."""
    
    if thickness_range is None:
        thickness_range = np.logspace(-2, 2, 100)  # 0.01 to 100 cm
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(energies_to_plot)))
    
    density = MATERIAL_DENSITIES.get(isotope, 1.0)
    
    for energy_ev, color in zip(energies_to_plot, colors):
        try:
            prob_vs_thickness = []
            for thickness in thickness_range:
                prob, xs, macro_xs = calculate_capture_probability(
                    [energy_ev], isotope, nuclear_db, density, thickness, temperature_k)
                prob_vs_thickness.append(prob[0])
            
            plt.semilogx(thickness_range, prob_vs_thickness,
                        label=f'{energy_ev} eV', color=color, linewidth=2.5)
            
        except Exception as e:
            print(f"Warning: Could not plot energy {energy_ev}: {e}")
    
    plt.xlabel('Thickness (cm)', fontsize=14)
    plt.ylabel('Capture Probability', fontsize=14)
    plt.title(f'Capture Probability vs Thickness - {isotope}', fontsize=16)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

def plot_temperature_dependence(isotope, energies, temperatures=[77, 300, 600, 1000],
                              thickness_cm=DEFAULT_THICKNESS):
    """Plot how temperature affects capture probability."""
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.plasma(np.linspace(0, 1, len(temperatures)))
    density = MATERIAL_DENSITIES.get(isotope, 1.0)
    
    for temp_k, color in zip(temperatures, colors):
        try:
            prob, xs, macro_xs = calculate_capture_probability(
                energies, isotope, nuclear_db, density, thickness_cm, temp_k)
            
            plt.loglog(energies, prob, label=f'{temp_k} K',
                      color=color, linewidth=2.5)
            
        except Exception as e:
            print(f"Warning: Could not plot temperature {temp_k}: {e}")
    
    plt.xlabel('Neutron Energy (eV)', fontsize=14)
    plt.ylabel('Capture Probability', fontsize=14)
    plt.title(f'Temperature Dependence - {isotope} ({thickness_cm} cm thick)', fontsize=16)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def interactive_comparison():
    """Interactive function to compare materials at user-specified energy."""
    print("\nInteractive Material Comparison")
    print("Available materials:", SELECTED_MATERIALS)
    
    try:
        energy_str = input("Enter neutron energy (eV): ")
        energy_ev = float(energy_str)
        
        thickness_str = input(f"Enter thickness (cm, default {DEFAULT_THICKNESS}): ")
        thickness_cm = float(thickness_str) if thickness_str else DEFAULT_THICKNESS
        
        results = compare_materials_at_energy(energy_ev, SELECTED_MATERIALS, nuclear_db, thickness_cm)
        
        # Plot the results
        materials = [r[0] for r in results]
        probabilities = [r[1] for r in results]
        
        plt.figure(figsize=(10, 6))
        bars = plt.bar(materials, probabilities, color=plt.cm.viridis(np.linspace(0, 1, len(materials))))
        plt.ylabel('Capture Probability', fontsize=12)
        plt.title(f'Material Comparison at {energy_ev} eV ({thickness_cm} cm thick)', fontsize=14)
        plt.xticks(rotation=45)
        
        # Add value labels on bars
        for bar, prob in zip(bars, probabilities):
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{prob:.4f}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error: {e}")

print("✓ Plotting functions loaded")

In [ ]:
# ===================================================================
# GENERATE ENERGY ARRAY AND SETUP
# ===================================================================

# Generate neutron energy array (logarithmic spacing)
energies = np.logspace(np.log10(ENERGY_MIN), np.log10(ENERGY_MAX), NUM_POINTS)

print(f"Generated {NUM_POINTS} energy points from {ENERGY_MIN:.1e} to {ENERGY_MAX:.1e} eV")
print(f"Temperature: {TEMPERATURE} K")
print(f"Default thickness: {DEFAULT_THICKNESS} cm")

# Selected materials info
print(f"\nSelected materials for analysis:")
print("-" * 50) 
for i, isotope in enumerate(SELECTED_MATERIALS, 1):
    try:
        density = MATERIAL_DENSITIES.get(isotope, 'N/A')
        mass, thermal_xs, abundance, resonances, fast_xs = nuclear_db.get_isotope_data(isotope)
        print(f"{i:2d}. {isotope:8s} | σ_th = {thermal_xs:8.1f} b | ρ = {density:6.3f} g/cm³")
    except Exception as e:
        print(f"{i:2d}. {isotope:8s} - Error: {e}")

print(f"\n✓ Ready to calculate neutron capture for {len(SELECTED_MATERIALS)} isotopes!")

In [ ]:
# ===================================================================
# PLOT 1: NEUTRON CAPTURE CROSS-SECTIONS
# ===================================================================

# Plot cross-sections vs energy for all selected materials
print("Plotting neutron capture cross-sections...")
plot_cross_sections(SELECTED_MATERIALS, energies, TEMPERATURE)

In [ ]:
# ===================================================================
# PLOT 2: CAPTURE PROBABILITIES VS ENERGY
# ===================================================================

# Plot capture probabilities for different materials
print("Plotting neutron capture probabilities...")
plot_capture_probabilities(SELECTED_MATERIALS, energies, DEFAULT_THICKNESS, TEMPERATURE)

In [ ]:
# ===================================================================
# PLOT 3: THICKNESS DEPENDENCE EXAMPLE
# ===================================================================

# Example: Boron-10 thickness dependence at different energies
example_energies = [0.01, 0.0253, 0.1, 1.0, 100.0]  # eV 

print("Plotting thickness dependence for B-10...")
plot_thickness_dependence('B-10', example_energies)

In [ ]:
# ===================================================================
# PLOT 4: TEMPERATURE DEPENDENCE EXAMPLE
# ===================================================================

# Example: Gadolinium-157 temperature dependence
print("Plotting temperature dependence for Gd-157...")
plot_temperature_dependence('Gd-157', energies, [77, 300, 600, 1000], DEFAULT_THICKNESS)

In [ ]:
# ===================================================================
# EXAMPLE CALCULATIONS
# ===================================================================

# Specific calculation examples
print("EXAMPLE SPECIFIC CALCULATIONS")
print("=" * 70)

# Example cases: (isotope, energy_eV, thickness_cm)
example_cases = [
    ("Gd-157", 0.0253, 1.0),    # Thermal neutrons in Gd
    ("B-10", 0.0253, 0.1),      # Thermal neutrons in thin B layer  
    ("Cd-113", 0.178, 2.0),     # Resonance energy in Cd
    ("He-3", 0.0253, 10.0),     # Thermal neutrons in He-3 detector
    ("U-235", 0.29, 5.0),       # Near resonance in U-235
]

for isotope, energy, thickness in example_cases:
    calculate_specific_neutron_case(isotope, energy, nuclear_db, thickness_cm=thickness)

In [ ]:
# ===================================================================
# MATERIAL COMPARISON AT SPECIFIC ENERGIES
# ===================================================================

# Compare all materials at thermal energy
print("\nMaterial comparison at thermal energy (0.0253 eV):")
thermal_results = compare_materials_at_energy(0.0253, SELECTED_MATERIALS, nuclear_db, 1.0)

print("\nMaterial comparison at fast neutron energy (1 MeV):")
fast_results = compare_materials_at_energy(1e6, SELECTED_MATERIALS, nuclear_db, 1.0)

In [ ]:
# ===================================================================
# USAGE INSTRUCTIONS AND CUSTOMIZATION
# ===================================================================

print("\n" + "="*80)
print("NEUTRON CAPTURE CALCULATOR - READY FOR USE!")
print("="*80)

print("\n🎯 HOW TO USE THIS NOTEBOOK:")
print("1. Modify SELECTED_MATERIALS list to choose isotopes")
print("2. Adjust MATERIAL_DENSITIES, THICKNESSES, and energy range") 
print("3. Run all cells to generate comprehensive plots")
print("4. Use specific calculation functions for detailed analysis")

print("\n📚 AVAILABLE FUNCTIONS:")
print("• calculate_capture_probability(energies, isotope, density, thickness, temp)")
print("• calculate_specific_case(isotope, energy_ev, density, thickness, temp)")
print("• compare_materials_at_energy(energy_ev, materials_list, thickness)")
print("• plot_cross_sections(materials_list, energies, temperature)")
print("• plot_capture_probabilities(materials_list, energies, thickness, temp)")
print("• plot_thickness_dependence(isotope, energies_list, thickness_range)")
print("• plot_temperature_dependence(isotope, energies, temperatures, thickness)")
print("• interactive_comparison()")

print("\n🧪 CONFIGURABLE PARAMETERS:")
print("• Material/Isotope: Choose from nuclear database or add new ones")
print("• Density: Material density in g/cm³")  
print("• Thickness: Absorber thickness in cm")
print("• Temperature: Affects 1/v law for thermal neutrons (Kelvin)")
print("• Energy Range: From ultra-cold to fast neutrons (eV)")
print("• Abundance: Override natural isotopic abundances")

print("\n⚛️ PHYSICS INCLUDED:")
print("• 1/v law for thermal neutrons with temperature dependence")
print("• Breit-Wigner resonances for major peaks")
print("• Smooth transition to fast neutron cross-sections")  
print("• Full Beer-Lambert attenuation: P = 1 - exp(-Σt)")
print("• Accurate number densities: n = ρ·N_A·abundance/A")

print("\n🚀 TO ADD NEW ISOTOPES:")
print("• Add entry to nuclear_data dictionary in NuclearDatabase class")
print("• Format: 'Isotope': (mass, σ_thermal, abundance, resonances, σ_fast)")
print("• Resonances: [(E_res, Γ_n, Γ_γ, J), ...] for Breit-Wigner peaks")

print("\n💡 EXAMPLE USAGE:")
print("# Calculate B-10 capture at thermal energy")
print("prob, xs, macro_xs = calculate_capture_probability([0.0253], 'B-10', 2.34, 0.1)")
print("print(f'Capture probability: {prob[0]:.4f}')")

print("\n# Interactive comparison")
print("# interactive_comparison()  # Uncomment to run")

print("\n" + "="*80)
print("🔬 Perfect for neutron detector design, shielding calculations,")
print("   reactor physics, and nuclear engineering applications!")
print("="*80)